In [ ]:
import pandas as pd
import numpy as np

df=pd.read_csv('/content/spotify_tracks.csv')
df_indian=df[df['language'].isin(['Hindi',  'Unknown'])].copy()

In [ ]:
df_indian.reset_index(drop=True,inplace=True)

In [ ]:
df_indian.head(1)

,track_id,track_name,artist_name,year,popularity,artwork_url,album_name,acousticness,danceability,duration_ms,...,key,liveness,loudness,mode,speechiness,tempo,time_signature,valence,track_url,language
0,7qjOhCEl3pxRjJ2mHnfGrs,"Villain Kaun Hai (From ""Leo (Hindi)"")","Anirudh Ravichander, Manisha Eerabathini, Samp...",2024,34,https://i.scdn.co/image/ab67616d0000b273244603...,"Villain Kaun Hai [From ""Leo (Hindi)""]",0.28,0.684,185802.0,...,7.0,0.0818,-8.282,0.0,0.0419,106.031,4.0,0.451,https://open.spotify.com/track/7qjOhCEl3pxRjJ2...,Hindi


In [ ]:
columns_to_keep = ['track_name', 'artist_name', 'language', 'danceability', 'energy', 'acousticness', 'tempo', 'valence']
df_indian = df_indian[columns_to_keep]

In [ ]:
df_indian.head(1)

,track_name,artist_name,language,danceability,energy,acousticness,tempo,valence
0,"Villain Kaun Hai (From ""Leo (Hindi)"")","Anirudh Ravichander, Manisha Eerabathini, Samp...",Hindi,0.684,0.772,0.28,106.031,0.451


In [ ]:
df_indian.dropna(inplace=True)
df_indian.reset_index(drop=True,inplace=True)

In [ ]:
features=df_indian[['danceability', 'energy', 'acousticness', 'tempo', 'valence']]

In [ ]:
features.head(1)

,danceability,energy,acousticness,tempo,valence
0,0.684,0.772,0.28,106.031,0.451


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
scaled_features=scaler.fit_transform(features)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
similarity=cosine_similarity(scaled_features)

In [ ]:
similarity[1]

array([ 0.90963882,  1.        ,  0.9488924 , ...,  0.36614866,
       -0.49759552,  0.69740916])

In [ ]:
def recommend_song(song_name):
  if song_name not in df_indian['track_name'].values:
    return 'Song not found ! '

In [ ]:
song_name = 'Villain Kaun Hai (From "Leo (Hindi)")' # Example: Define a song_name for testing
idx = df_indian[df_indian['track_name'] == song_name].index[0]

In [ ]:
input_lang = df_indian.iloc[idx]['language']
input_artist = df_indian.iloc[idx]['artist_name']
print(f"🔎 Searching matches for: '{song_name}' by {input_artist} (Language: {input_lang})\n")

🔎 Searching matches for: 'Villain Kaun Hai (From "Leo (Hindi)")' by Anirudh Ravichander, Manisha Eerabathini, Sampath GK (Language: Hindi)



In [ ]:
distances = similarity[idx]

array([ 1.        ,  0.90963882,  0.85243557, ...,  0.35447682,
       -0.41175991,  0.59346386])

In [ ]:
song_indices=sorted(list(enumerate(distances)),reverse=True,key=lambda x:x[1])[1:6]

In [ ]:
result=[]
for i in song_indices:
  index=i[0]
  match_score=round(i[1]*100,2)

  songq=df_indian.iloc[index]['track_name']
  artistq=df_indian.iloc[index]['artist_name']
  Languagereq=df_indian.iloc[index]['language']
  match_score=round(i[1]*100,2)

  result.append((songq,artistq,Languagereq,match_score))



In [ ]:
def recommend_song(song_name):
    if song_name not in df_indian['track_name'].values:
        return []

    # 1. Find the index of the song
    idx = df_indian[df_indian['track_name'] == song_name].index[0]

    # 2. Get similarity scores
    distances = similarity[idx]

    # 3. Sort and get top 5 (excluding the song itself)
    song_indices = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]

    # 4. Format the results
    res = []
    for i in song_indices:
        index = i[0]
        res.append({
            'Song': df_indian.iloc[index]['track_name'],
            'Singer': df_indian.iloc[index]['artist_name'],
            'Language': df_indian.iloc[index]['language'],
            'Match_Score': round(i[1] * 100, 2)
        })
    return res

# Now test the function
recommendations = recommend_song("Villain Kaun Hai (From \"Leo (Hindi)\")")
if not recommendations:
    print("Song not found!")
else:
    for r in recommendations:
        print(f"🎵 {r['Song']} | 🎤 {r['Singer']} | 🌐 {r['Language']} | ⚡ Match: {r['Match_Score']}")

🎵 Tajuddin Shahenshah Sandal Tera | 🎤 Ali Javed Warsi | 🌐 Unknown | ⚡ Match: 99.54
🎵 Jack And Jill | 🎤 A.R. Rahman, Shurjo Bhattacharya, Nithyasree Mahadevan, Mathangi Jagdish | 🌐 Unknown | ⚡ Match: 99.11
🎵 Sher Punjabi | 🎤 Vidyasagar, Rehaan Khan | 🌐 Unknown | ⚡ Match: 98.94
🎵 Micromax Unite Cricket Anthem | 🎤 Benny Dayal, Hari & Sukhmani, Akriti Kakar, Karthik, Shalmali Kholgade, Zubeen Garg, Raghu Dixit, Kavita Seth, Shakthisree Gopalan | 🌐 Unknown | ⚡ Match: 98.94
🎵 Dhak Baja Kashor Baja (From "Dhak Baja Kashor Baja") | 🎤 Shreya Ghoshal | 🌐 Unknown | ⚡ Match: 98.9


In [ ]:
import pickle

pickle.dump(df_indian,open('indian_song.pkl','wb'))
pickle.dump(similarity,open('similarity.pkl','wb'))